In [6]:
from pathlib import Path

Path.cwd()


PosixPath('/Users/muhabahmad/LandingSite/Lunar-Subsurface-Ice-Detection/Landing/notebooks')

In [7]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "Data1"
PROCESSED_DATA = PROJECT_ROOT / "outputs"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROCESSED_DATA:", PROCESSED_DATA)

PROJECT_ROOT: /Users/muhabahmad/LandingSite/Lunar-Subsurface-Ice-Detection/Landing
DATA_DIR: /Users/muhabahmad/LandingSite/Lunar-Subsurface-Ice-Detection/Landing/Data1
PROCESSED_DATA: /Users/muhabahmad/LandingSite/Lunar-Subsurface-Ice-Detection/Landing/outputs


In [8]:
list(PROCESSED_DATA.glob("*"))

[]

In [9]:
PROCESSED_DATA.mkdir(exist_ok=True)

In [11]:
for file in PROJECT_ROOT.rglob("filtered_crops.npy"):
    print(file)

for file in PROJECT_ROOT.rglob("filtered_locations.npy"):
    print(file)

/Users/muhabahmad/LandingSite/Lunar-Subsurface-Ice-Detection/Landing/data/processed/filtered_crops.npy
/Users/muhabahmad/LandingSite/Lunar-Subsurface-Ice-Detection/Landing/data/processed/filtered_locations.npy


In [12]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

filtered_crops = np.load(PROCESSED_DATA / "filtered_crops.npy")
filtered_locations = np.load(PROCESSED_DATA / "filtered_locations.npy")

print(filtered_crops.shape)
print(filtered_locations.shape)

(89, 1000, 1000)
(89, 2)


In [13]:
results = []

for i, (crop, loc) in enumerate(zip(filtered_crops, filtered_locations)):
    features = extract_features(crop)
    features["crop_id"] = i
    features["pixel_row"] = loc[0]
    features["pixel_col"] = loc[1]
    results.append(features)

df = pd.DataFrame(results)

print(df.head())
print(df.describe())

df.to_csv(PROCESSED_DATA / "crop_features.csv", index=False)

   mean_brightness  std_brightness  shadow_percentage  roughness  \
0        12.707701        6.643544            86.9905   0.003368   
1        15.789172       10.411407            67.1195   0.004227   
2        13.837310       12.797110            70.8379   0.004550   
3        23.446335       17.382391            47.0601   0.005885   
4        17.461168       14.965025            61.6870   0.008044   

   max_slope_proxy  crop_id  pixel_row  pixel_col  
0         0.041595        0          0       4000  
1         0.060879        1          0       5000  
2         0.069629        2          0       6000  
3         0.072655        3          0       7000  
4         0.096557        4          0       8000  
       mean_brightness  std_brightness  shadow_percentage  roughness  \
count        89.000000       89.000000          89.000000  89.000000   
mean         25.609819       15.430045          51.316031   0.006715   
std          17.430542        6.544546          31.427949   0.0

In [14]:
# Normalize features

df["shadow_norm"] = df["shadow_percentage"] / 100
df["roughness_norm"] = df["roughness"] / df["roughness"].max()
df["slope_norm"] = df["max_slope_proxy"] / df["max_slope_proxy"].max()

# Composite landing score
df["landing_score"] = (
    0.40 * (1 - df["shadow_norm"]) +
    0.30 * (1 - df["roughness_norm"]) +
    0.30 * (1 - df["slope_norm"])
)

# Rank
df_ranked = df.sort_values("landing_score", ascending=False)

display(
    df_ranked[
        [
            "crop_id",
            "landing_score",
            "shadow_percentage",
            "roughness",
            "max_slope_proxy",
            "pixel_row",
            "pixel_col",
        ]
    ].head(15)
)

df_ranked.to_csv(
    PROCESSED_DATA / "landing_candidates_ranked.csv",
    index=False
)

,crop_id,landing_score,shadow_percentage,roughness,max_slope_proxy,pixel_row,pixel_col
80,80,0.612123,1.9208,0.008246,0.096398,14000,4000
86,86,0.610053,0.6891,0.008570,0.096258,15000,5000
79,79,0.607089,19.7814,0.006487,0.083097,14000,3000
85,85,0.603094,36.6346,0.004438,0.074510,15000,4000
66,66,0.600439,5.7517,0.008520,0.091550,12000,3000
71,71,0.591517,25.0854,0.006125,0.084337,13000,1000
73,73,0.585387,1.8293,0.009093,0.100384,13000,3000
47,47,0.576122,12.8599,0.008060,0.094627,9000,1000
52,52,0.563157,12.6929,0.008274,0.098995,10000,1000
49,49,0.560692,11.3851,0.008734,0.097686,9000,3000


In [15]:
safe = df_ranked[
    (df_ranked["shadow_percentage"] < 80) &
    (df_ranked["max_slope_proxy"] < 0.30)
]

print(f"Safe candidates: {len(safe)}")
display(safe.head(10))

Safe candidates: 65


,mean_brightness,std_brightness,shadow_percentage,roughness,max_slope_proxy,crop_id,pixel_row,pixel_col,shadow_norm,roughness_norm,slope_norm,landing_score
80,59.451351,19.312899,1.9208,0.008246,0.096398,80,14000,4000,0.019208,0.617557,0.649756,0.612123
86,59.989128,17.496786,0.6891,0.008570,0.096258,86,15000,5000,0.006891,0.641821,0.648815,0.610053
79,35.294384,17.027906,19.7814,0.006487,0.083097,79,14000,3000,0.197814,0.485851,0.560100,0.607089
85,25.325184,13.427577,36.6346,0.004438,0.074510,85,15000,4000,0.366346,0.332337,0.502222,0.603094
66,47.493923,17.336535,5.7517,0.008520,0.091550,66,12000,3000,0.057517,0.638103,0.617079,0.600439
71,28.317076,12.450066,25.0854,0.006125,0.084337,71,13000,1000,0.250854,0.458681,0.568458,0.591517
73,66.204903,20.552576,1.8293,0.009093,0.100384,73,13000,3000,0.018293,0.681031,0.676621,0.585387
47,50.055614,22.925425,12.8599,0.008060,0.094627,47,9000,1000,0.128599,0.603644,0.637818,0.576122
52,41.796062,18.538803,12.6929,0.008274,0.098995,52,10000,1000,0.126929,0.619640,0.667263,0.563157
49,44.600208,18.999704,11.3851,0.008734,0.097686,49,9000,3000,0.113851,0.654122,0.658435,0.560692


In [16]:
print(safe.shape)

(65, 12)
